# Time Series Prediction with PyTorch

In [ ]:
import random
import time

import lightning.pytorch as pl
import pandas as pd
import ray
from matplotlib import pyplot as plt

from pytorch_forecasting import DeepAR, TimeSeriesDataSet
from pytorch_forecasting.data import NaNLabelEncoder
from pytorch_forecasting.data.examples import generate_ar_data
from pytorch_forecasting.metrics import NormalDistributionLoss

In [ ]:
if ray.is_initialized():
    ray.shutdown()

display(ray.init(address="auto"))
display(ray.cluster_resources())

In [ ]:
total_gpus = 1  # does not work with more than 1!
total_cpus = 8

Generate synthetic time series data.

---

In [ ]:
data = generate_ar_data(seasonality=10.0, timesteps=400, n_series=2000, seed=42)
data["static"] = 2
data["date"] = pd.Timestamp("2020-01-01") + pd.to_timedelta(data.time_idx, "D")
data["series"] = data["series"].astype(str)
data["static"] = data["static"].astype(str)

display(data.head())

In [ ]:
plt.figure(figsize=(10, 6))

for series_id in data.series.unique()[:10]:
    series_data = data[data.series == series_id]
    plt.plot(
        series_data.date,
        series_data.value,
        label=f"Series {series_id}",
    )

plt.legend()
plt.title("Generated Time Series Data")
plt.xlabel("Date")
plt.ylabel("Value")
plt.show()

Create the training and validation datasets using `TimeSeriesDataSet`. The validation set is generated using `from_dataset()` to ensure the categorical encoders and scalers perfectly match the training set.

---

In [ ]:
max_encoder_length = 60
max_prediction_length = 20
training_cutoff = data["time_idx"].max() - max_prediction_length

training = TimeSeriesDataSet(
    data[lambda x: x.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="value",
    group_ids=["series"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    static_categoricals=["series", "static"],
    time_varying_known_reals=["time_idx"],
    time_varying_unknown_reals=["value"],
    categorical_encoders={"series": NaNLabelEncoder().fit(data.series)},
)

validation = TimeSeriesDataSet.from_dataset(
    training,
    data,
    min_prediction_idx=training.index.time.max() + 1,
    stop_randomization=True,
)

In [ ]:
batch_size = 256
baseline_workers = total_cpus - 1

train_dataloader = training.to_dataloader(
    train=True,
    batch_size=batch_size,
    num_workers=baseline_workers,
    persistent_workers=True,
)
val_dataloader = validation.to_dataloader(
    train=False,
    batch_size=batch_size,
    num_workers=baseline_workers,
    persistent_workers=True,
)

Instantiate a model using the `.from_dataset()` method.

---

In [ ]:
pl.seed_everything(42, workers=True)

deepar = DeepAR.from_dataset(
    training,
    learning_rate=1e-3,
    hidden_size=64,
    rnn_layers=3,
    dropout=0.1,
    loss=NormalDistributionLoss(),
    reduce_on_plateau_patience=4,
)
print(f"Number of parameters in network: {deepar.size() / 1e3:.1f}k")

Train the model. Early stopping and checkpointing are disabled to ensure a strict 10-epoch performance benchmark.

---

In [ ]:
trainer = pl.Trainer(
    max_epochs=10,
    accelerator="gpu",
    devices=total_gpus,
    gradient_clip_val=0.1,
    enable_checkpointing=False,
    logger=False,
    enable_progress_bar=True,
)

start_time = time.time()

trainer.fit(deepar, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

print(f"Training Time: {time.time() - start_time:.2f} seconds")

Plot a sample of results. We sample 5 series prior to generating predictions.

---

In [ ]:
best_deepar = deepar

sample_series_ids = random.sample(list(data["series"].unique()), 5)

sample_data = data[data["series"].isin(sample_series_ids)]

raw_predictions = best_deepar.predict(sample_data, mode="raw", return_x=True)

for idx in range(5):
    fig, ax = plt.subplots(figsize=(10, 5))
    best_deepar.plot_prediction(
        raw_predictions.x,
        raw_predictions.output,
        idx=idx,
        add_loss_to_title=True,
        ax=ax,
    )
    plt.show()